In [2]:
import tomllib
import pandas as pd
from pathlib import Path

In [5]:
with open("permissible_sites.toml","rb") as f:
    data = tomllib.load(f)

In [ ]:
data["meta"]

{'generated_at': '2026-04-16T01:40:53+00:00',
 'tool_version': '1.2.0',
 'total_checked': 96,
 'allowed': 41,
 'allowed_via_api': 12,
 'manual_review': 30,
 'serp_only': 12,
 'blocked': 0,
 'unreachable': 1}

In [ ]:
review = pd.DataFrame()
sites = data["sites"]
for verdict in sites.keys():
    thisdf = pd.DataFrame(sites[verdict])
    thisdf["verdict"] = verdict
    review = pd.concat([review, thisdf], axis=0, ignore_index=True)

# replace empty signal records with NA
review["block_signals"] = review["block_signals"].apply(lambda x: pd.NA if len(x)==0 else x)
review["allow_signals"] = review["allow_signals"].apply(lambda x: pd.NA if len(x)==0 else x)

In [ ]:
# building a conditional mask, where a explicitly permits robotic access
robots_did_not_deny = (review["robots_denies"]!=True)                           # does not prohibit
robot_allows = (review["robots_status"]=="found") & (review["robots_allows"])   # explicitly permits
block_signs_not_found = review["block_signals"].isna()                          # does not block
allow_signs_found = (review["allow_signals"].isna()!=True)                      # explicitly permits

# club all masks
masks = robots_did_not_deny & robot_allows & block_signs_not_found & allow_signs_found
# review columns
cols = ["id", "base_url", "verdict", "robots_status", "robots_allows", "robots_denies", "allow_signals", "allow_signal_contexts", "block_signals"]

# review the data
review.loc[masks, cols]

,id,base_url,verdict,robots_status,robots_allows,robots_denies,allow_signals,allow_signal_contexts,block_signals
3,freefincal,https://freefincal.com,allowed,found,True,False,[non-profit],{'non-profit': '...pically includes online cou...,<NA>
31,bis_research,https://www.bis.org,allowed,found,True,False,"[non-commercial, may reproduce]",{'non-commercial': '...conditions without obta...,<NA>
40,livemint,https://www.livemint.com,allowed,found,True,False,[non-commercial],{'non-commercial': '...llowed to do (licence) ...,<NA>


### each site that was computed as "allowed" verdict seems to permit only non-commercial purposes.
### this is kind of a red flag where the data owner is not comfortable sharing his content, and it is obvious
### better to start resource search manullay from scratch